___

___

Esta unidad se divide en dos secciones:
* Primero, descubriremos qué se necesita para construir un sistema NLP que pueda convertir un cuerpo de texto en una matriz numérica de *características*.
* Luego mostraremos cómo realizar estos pasos utilizando herramientas reales.

# Construyendo un procesador de lenguaje natural desde cero
En esta sección, usaremos Python básico para construir un sistema NLP rudimentario. Construiremos un *corpus de documentos* (dos pequeños archivos de texto), crearemos un *vocabulario* de todas las palabras en ambos documentos y luego demostraremos una técnica de *Bolsa de palabras* para extraer características de cada documento.<br>
<div class="alert alert-info" style="margin: 20px">**¡Esta primera sección es solo ilustrativa!**
<br>No se moleste en memorizar el código, nunca haríamos esto en la vida real.</div>

## Comience con algunos documentos:
Para simplificar, no usaremos ningún signo de puntuación.

In [15]:
%%writefile 1.txt
Esta es una historia de gatos
nuestras mascotas felinas
Los gatos son animales peludos

Overwriting 1.txt


In [16]:
%%writefile 2.txt
Esta historia es sobre el surf
Surfear sobre las olas es divertido
El surf es un deporte acuático popular

Overwriting 2.txt


## Construye un vocabulario
El objetivo aquí es construir una matriz numérica a partir de todas las palabras que aparecen en cada documento. Luego crearemos instancias (vectores) para cada documento individual.

In [17]:
vocab = {}
i = 1

with open('1.txt') as f:
    x = f.read().lower().split()

for word in x:
    if word in vocab:
        continue
    else:
        vocab[word]=i
        i+=1

print(vocab)

{'esta': 1, 'es': 2, 'una': 3, 'historia': 4, 'de': 5, 'gatos': 6, 'nuestras': 7, 'mascotas': 8, 'felinas': 9, 'los': 10, 'son': 11, 'animales': 12, 'peludos': 13}


In [18]:
with open('2.txt') as f:
    x = f.read().lower().split()

for word in x:
    if word in vocab:
        continue
    else:
        vocab[word]=i
        i+=1

print(vocab)

{'esta': 1, 'es': 2, 'una': 3, 'historia': 4, 'de': 5, 'gatos': 6, 'nuestras': 7, 'mascotas': 8, 'felinas': 9, 'los': 10, 'son': 11, 'animales': 12, 'peludos': 13, 'sobre': 14, 'el': 15, 'surf': 16, 'surfear': 17, 'las': 18, 'olas': 19, 'divertido': 20, 'un': 21, 'deporte': 22, 'acuático': 23, 'popular': 24}


Aunque `2.txt` tiene 19 palabras, solo se agregaron 12 palabras nuevas al diccionario.

## Extracción de características
Ahora que hemos encapsulado nuestro "lenguaje completo" en un diccionario, realicemos *extracción de funciones* en cada uno de nuestros documentos originales:

In [19]:
# Crea un vector vacío con espacio para cada palabra en el vocabulario:
one = ['1.txt']+[0]*len(vocab)
print(one)

['1.txt', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [20]:
# mapear las frecuencias de cada palabra en 1.txt a nuestro vector:
with open('1.txt') as f:
    x = f.read().lower().split()
    
for word in x:
    one[vocab[word]]+=1
    
print(one)

['1.txt', 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


<font color=green>Podemos ver que la mayoría de las palabras en 1.txt aparecen solo una vez, aunque "gatos" aparece dos veces.</font>

In [21]:
# Hacemos lo mismo para el segundo documento:
two = ['2.txt']+[0]*len(vocab)

with open('2.txt') as f:
    x = f.read().lower().split()
    
for word in x:
    two[vocab[word]]+=1

In [22]:
# Compare the two vectors:
print(f'{one}\n{two}')

['1.txt', 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
['2.txt', 1, 3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1]


Al comparar los vectores vemos que algunas palabras son comunes a ambos, algunas aparecen solo en `1.txt`, otras solo en `2.txt`. Extendiendo esta lógica a decenas de miles de documentos, veríamos crecer el diccionario de vocabulario a cientos de miles de palabras. Los vectores contendrían en su mayoría valores cero, haciéndolos *matrices dispersas*.

## Bolsa de Palabras y Tf-idf
En los ejemplos anteriores, cada vector puede considerarse una *bolsa de palabras*. Por sí mismos, estos pueden no ser útiles hasta que consideremos *frecuencias de términos*, o con qué frecuencia aparecen palabras individuales en los documentos. Una forma simple de calcular la frecuencia de los términos es dividir el número de ocurrencias de una palabra por el número total de palabras en el documento. De esta manera, la cantidad de veces que aparece una palabra en documentos grandes se puede comparar con la de documentos más pequeños.

Sin embargo, puede ser difícil diferenciar los documentos según la frecuencia de los términos si una palabra aparece en la mayoría de los documentos. Para manejar esto, también consideramos la *frecuencia inversa de documentos*, que es el número total de documentos dividido por el número de documentos que contienen la palabra. En la práctica, convertimos este valor a una escala logarítmica, como se describe [aquí](https://en.wikipedia.org/wiki/Tf%E2%80%93idf#Inverse_document_frequency).

Juntos, estos términos se convierten en [**tf-idf**](https://en.wikipedia.org/wiki/Tf%E2%80%93idf).

## Stop words y raíces de palabras
Algunas palabras como "el" y "y" aparecen con tanta frecuencia y en tantos documentos que no necesitamos molestarnos en contarlas. Además, puede tener sentido registrar solo la raíz de una palabra, decir 'gato' en lugar de 'gato' y 'gatos'. Esto reducirá nuestra matriz de vocabulario y mejorará el rendimiento.

## Tokenización y etiquetado
Cuando creamos nuestros vectores, lo primero que hicimos fue dividir el texto entrante en espacios en blanco con `.split()`. Esta era una forma cruda de *tokenización*, es decir, dividir un documento en palabras individuales. En este ejemplo simple, no nos preocupamos por la puntuación o las diferentes partes del discurso. En el mundo real, confiamos en una *morfología* bastante sofisticada para analizar el texto de manera adecuada.

Una vez que se divide el texto, podemos regresar y *etiquetar* nuestros tokens con información sobre partes del discurso, dependencias gramaticales, etc. Esto agrega más dimensiones a nuestros datos y permite una comprensión más profunda del contexto de documentos específicos. Por esta razón, los vectores se convierten en ***matrices dispersas de alta dimensión***.

<div class="alert alert-info" style="margin: 20px">**Ese es el final de la primera sección.**
<br>En la siguiente sección, usaremos scikit-learn para realizar un análisis de la vida real.</div>

___
# Extracción de características del texto
En el anterior notebook **Scikit-learn Primer**, aplicamos un modelo de clasificación SVC simple al conjunto de datos de SMSSpamCollection. Intentamos predecir la etiqueta de ham/spam en función de la longitud del mensaje y la puntuación. En esta sección, veremos el texto de cada mensaje e intentaremos realizar una clasificación basada en el contenido. Aprovecharemos algunas de las herramientas de [extracción de características](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) de scikit-learn.

## Cargar un conjunto de datos

In [9]:
# Realizar importaciones y cargar el conjunto de datos:
import numpy as np
import pandas as pd

df = pd.read_csv('../datos/smsspamcollection.tsv', sep='\t')
df.head()

,label,message,length,punct
0,ham,"Go until jurong point, crazy.. Available only ...",111,9
1,ham,Ok lar... Joking wif u oni...,29,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,6
3,ham,U dun say so early hor... U c already then say...,49,6
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,2


## Comprobar si faltan valores:
Siempre una buena práctica.

In [10]:
df.isnull().sum()

label      0
message    0
length     0
punct      0
dtype: int64

## Eche un vistazo rápido a la columna `label` *ham* y *spam*:

In [11]:
df['label'].value_counts()

ham     4825
spam     747
Name: label, dtype: int64

<font color=green>4825 de 5572 mensajes, o el 86,6%, son ham. Esto significa que cualquier modelo de clasificación de texto que creemos tiene que funcionar **mejor que 86.6%** para vencer la probabilidad aleatoria.</font>

## Dividir los datos en conjuntos de entrenamiento y prueba:

In [12]:
from sklearn.model_selection import train_test_split

X = df['message']  # esta vez queremos mirar el texto
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## CountVectorizer de Scikit-learn
El preprocesamiento de texto, la tokenización y la capacidad de filtrar palabras vacías se incluyen en [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html), que crea un diccionario de características y transforma documentos en vectores de características.

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()

X_train_counts = count_vect.fit_transform(X_train)
X_train_counts.shape

(3733, 7082)

<font color=green>Esto muestra que nuestro conjunto de entrenamiento se compone de 3733 documentos y 7082 features.</font>

## Transformar conteos a frecuencias con Tf-idf
Si bien contar palabras es útil, los documentos más largos tendrán valores de conteo promedio más altos que los documentos más cortos, aunque puedan hablar sobre los mismos temas.

Para evitar esto, podemos simplemente dividir el número de ocurrencias de cada palabra en un documento por el número total de palabras en el documento: estas nuevas funciones se denominan **tf** para Frecuencias de términos.

Otro refinamiento además de **tf** es reducir el peso de las palabras que aparecen en muchos documentos del corpus y, por lo tanto, son menos informativas que las que aparecen solo en una porción más pequeña del corpus.

Esta reducción de escala se llama **tf–idf** para "Frecuencia de término por frecuencia de documento inversa".

Tanto tf como tf–idf se pueden calcular de la siguiente manera usando [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html):

In [14]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer = TfidfTransformer()

X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_train_tfidf.shape

(3733, 7082)

Nota: el método `fit_transform()` en realidad realiza dos operaciones: ajusta un estimador a los datos y luego transforma nuestra matriz de recuento en una representación tf-idf.

## Combinar pasos con TfidVectorizer
En el futuro, podemos combinar los pasos CountVectorizer y TfidTransformer en uno usando [TfidVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html):

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train) # recuerda usar el conjunto X_train original
X_train_tfidf.shape

(3733, 7082)

## Entrenar un Clasificador
Aquí presentaremos un clasificador que es similar a SVC, llamado [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html). LinearSVC maneja mejor la entrada dispersa y escala bien a un gran número de muestras.

In [16]:
from sklearn.svm import LinearSVC
clf = LinearSVC()
clf.fit(X_train_tfidf,y_train)

LinearSVC(C=1.0, class_weight=None, dual=True, fit_intercept=True,
     intercept_scaling=1, loss='squared_hinge', max_iter=1000,
     multi_class='ovr', penalty='l2', random_state=None, tol=0.0001,
     verbose=0)

<font color=green>Anteriormente llamamos a nuestro clasificador SVC **svc_model**. Aquí estamos usando el nombre más genérico **clf** (para clasificador).</font>

## Construya una tubería
Recuerde que solo nuestro conjunto de entrenamiento se ha vectorizado en un vocabulario completo. Para realizar un análisis en nuestro conjunto de prueba, tendremos que someterlo a los mismos procedimientos. Afortunadamente, scikit-learn ofrece una clase [**Pipeline**](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) que se comporta como un clasificador compuesto.

In [17]:
from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.svm import LinearSVC

text_clf = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', LinearSVC()),
])

# Alimente los datos de entrenamiento a través de la canalización
text_clf.fit(X_train, y_train)  

Pipeline(memory=None,
     steps=[('tfidf', TfidfVectorizer(analyzer='word', binary=False, decode_error='strict',
        dtype=<class 'numpy.float64'>, encoding='utf-8', input='content',
        lowercase=True, max_df=1.0, max_features=None, min_df=1,
        ngram_range=(1, 1), norm='l2', preprocessor=None, smooth_idf=True,...ax_iter=1000,
     multi_class='ovr', penalty='l2', random_state=None, tol=0.0001,
     verbose=0))])

## Probar el clasificador y mostrar los resultados

In [18]:
# Realiza un conjunto de predicciones
predictions = text_clf.predict(X_test)

In [19]:
# Reportar la matriz de confusión
from sklearn import metrics
print(metrics.confusion_matrix(y_test,predictions))

[[1586    7]
 [  12  234]]


In [20]:
# Imprimir un informe de clasificación
print(metrics.classification_report(y_test,predictions))

              precision    recall  f1-score   support

         ham       0.99      1.00      0.99      1593
        spam       0.97      0.95      0.96       246

   micro avg       0.99      0.99      0.99      1839
   macro avg       0.98      0.97      0.98      1839
weighted avg       0.99      0.99      0.99      1839



In [21]:
# Imprimir la precisión general
print(metrics.accuracy_score(y_test,predictions))

0.989668297988


Usando el texto de los mensajes, nuestro modelo funcionó muy bien; ¡predijo correctamente el spam el **98,97 %** de las veces!<br>
Ahora apliquemos lo que hemos aprendido a un proyecto de clasificación de texto que involucre críticas positivas y negativas de películas.

## A continuación: Proyecto de clasificación de texto